# CFPB Complaints: Preprocessing & Feature Engineering

## Notebook Contract

This notebook is the authoritative preprocessing workflow for modeling-ready data. It defines the final supervised target, applies leakage-safe transformations, and exports the train/test datasets used in modeling.

Model training and evaluation are done in Notebook 3.


## 1. Setup

Load the working sample, import shared policy logic, and establish deterministic preprocessing defaults.


In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import re

# Reproducibility for deterministic preprocessing steps.
np.random.seed(42)

# Make project package importable when running from notebooks/
sys.path.append(str(Path('..').resolve()))
from src.label_policy import build_supervised_target

# Load the sampled data
sample_file = Path('../data/processed/complaints_sample.csv')
df = pd.read_csv(sample_file)

print(f'Loaded {len(df):,} rows')
print(f'Columns: {list(df.columns)}')

Loaded 50,000 rows
Columns: ['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue', 'Consumer complaint narrative', 'Company public response', 'Company', 'State', 'ZIP code', 'Tags', 'Consumer consent provided?', 'Submitted via', 'Date sent to company', 'Company response to consumer', 'Timely response?', 'Consumer disputed?', 'Complaint ID']


## 2. Target Construction

Build the final binary target using the shared response-policy helper and remove ambiguous or unresolved outcomes.


In [2]:
# Define the binary target: monetary relief vs. no monetary relief
response_col = 'Company response to consumer'
print('Unique company responses (before filtering):')
print(df[response_col].value_counts(dropna=False))

eligible_mask, target_monetary_relief, policy_df = build_supervised_target(df[response_col])
excluded_count = (~eligible_mask).sum()
print(f"\nRows eligible for supervised target: {eligible_mask.sum():,} / {len(df):,}")
print(f"Rows excluded (ambiguous/unresolved response): {excluded_count:,}")

# Keep only rows with clear, closed outcomes for supervised modeling
df = df.loc[eligible_mask].copy()
policy_df = policy_df.loc[eligible_mask].copy()

# Create target from shared policy helper
df['target_monetary_relief'] = target_monetary_relief.values

print('\nEligible response breakdown used for target:')
print(df[response_col].value_counts(dropna=False))

print('\nTarget distribution:')
print(df['target_monetary_relief'].value_counts())
print(f"Class balance: {df['target_monetary_relief'].mean():.2%} positive")

# Sanity check by response category
print('\nPositive rate by response category (should be 1.00 only for monetary relief):')
print(df.groupby(response_col)['target_monetary_relief'].mean().sort_values(ascending=False))

Unique company responses (before filtering):
Company response to consumer
Closed with explanation            30033
Closed with non-monetary relief    17659
In progress                         1398
Closed with monetary relief          672
Untimely response                     91
Closed without relief                 68
Closed                                67
Closed with relief                    12
Name: count, dtype: int64

Rows eligible for supervised target: 48,499 / 50,000
Rows excluded (ambiguous/unresolved response): 1,501

Eligible response breakdown used for target:
Company response to consumer
Closed with explanation            30033
Closed with non-monetary relief    17659
Closed with monetary relief          672
Closed without relief                 68
Closed                                67
Name: count, dtype: int64

Target distribution:
target_monetary_relief
0    47827
1      672
Name: count, dtype: int64
Class balance: 1.39% positive

Positive rate by response category 

## 3. Feature Preparation

Clean narrative text, derive temporal fields, and encode selected structured variables for modeling.


In [3]:
# Helper function to clean narrative text
def clean_narrative(text):
    if pd.isna(text):
        return ''
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'\bx{2,}\b', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Clean narratives and create text-derived numeric features
df['narrative_clean'] = df['Consumer complaint narrative'].apply(clean_narrative)
df['narrative_length_raw'] = df['narrative_clean'].str.len()

# Outlier treatment: cap extreme narrative lengths at the 99th percentile
narrative_len_cap = df['narrative_length_raw'].quantile(0.99)
df['narrative_length'] = df['narrative_length_raw'].clip(upper=narrative_len_cap)
df['has_narrative'] = (df['narrative_length_raw'] > 0).astype(int)

print(f'Narratives present: {df["has_narrative"].sum()} / {len(df)}')
print(f'Narrative length (mean, raw): {df[df["has_narrative"] == 1]["narrative_length_raw"].mean():.0f} chars')
print(f'Narrative length cap (99th percentile): {narrative_len_cap:.0f} chars')
print(f'Narrative length (mean, capped): {df[df["has_narrative"] == 1]["narrative_length"].mean():.0f} chars')

Narratives present: 12723 / 48499
Narrative length (mean, raw): 941 chars
Narrative length cap (99th percentile): 3138 chars
Narrative length (mean, capped): 861 chars


In [4]:
# Extract temporal features
df['Date received'] = pd.to_datetime(df['Date received'], errors='coerce')
df['Date sent to company'] = pd.to_datetime(df['Date sent to company'], errors='coerce')

df['year_received'] = df['Date received'].dt.year
df['month_received'] = df['Date received'].dt.month
df['quarter_received'] = df['Date received'].dt.quarter
df['days_to_send'] = (df['Date sent to company'] - df['Date received']).dt.days

print('Temporal features created:')
print(f'  year_received: {df["year_received"].unique()}')
print(f'  days_to_send (mean): {df["days_to_send"].mean():.1f}')

Temporal features created:
  year_received: [2026 2025 2023 2024 2020 2019 2021 2022 2015 2016 2018 2014 2017 2012
 2013 2011]
  days_to_send (mean): 0.7


In [5]:
# Encode categorical features
categorical_features = ['Product', 'Issue', 'State', 'Submitted via']

# One-hot encode (we'll use label encoding later in the pipeline for tree models)
df_cat_encoded = pd.get_dummies(df[categorical_features], prefix=categorical_features, drop_first=False)

print(f'Categorical features encoded:')
print(f'  Total new columns: {len(df_cat_encoded.columns)}')
print(f'  Sample columns: {list(df_cat_encoded.columns)[:5]}')

Categorical features encoded:
  Total new columns: 243
  Sample columns: ['Product_Bank account or service', 'Product_Checking or savings account', 'Product_Consumer Loan', 'Product_Credit card', 'Product_Credit card or prepaid card']


## 4. Final Feature Set

Assemble numeric, categorical, and text-ready predictors into a single modeling table.


In [6]:
# Build the final feature set
# Numeric features
numeric_features = ['narrative_length', 'has_narrative', 'year_received', 'month_received', 'quarter_received', 'days_to_send']
X_numeric = df[numeric_features].fillna(0)

# Combine structured + text
X = pd.concat([X_numeric, df_cat_encoded, df[['narrative_clean']]], axis=1)
y = df['target_monetary_relief']

print(f'Feature matrix shape: {X.shape}')
print(f'Target shape: {y.shape}')
print(f'Target distribution:\n{y.value_counts()}')

Feature matrix shape: (48499, 250)
Target shape: (48499,)
Target distribution:
target_monetary_relief
0    47827
1      672
Name: count, dtype: int64


## 5. Documentation Artifacts

Export feature definitions and preprocessing summaries, including leakage checks and missingness diagnostics.


In [7]:
# Build and save a compact feature dictionary
reports_dir = Path('../reports')
reports_dir.mkdir(parents=True, exist_ok=True)

feature_dictionary_rows = []
for col in X.columns:
    if col == 'narrative_clean':
        source = 'Consumer complaint narrative'
        transform = 'text cleaned; vectorized later in modeling pipeline'
        feature_type = 'text'
    elif col in numeric_features:
        if col == 'narrative_length':
            source = 'Consumer complaint narrative'
            transform = 'character length with 99th percentile clipping'
        elif col == 'has_narrative':
            source = 'Consumer complaint narrative'
            transform = 'binary indicator for non-empty narrative'
        elif col in ['year_received', 'month_received', 'quarter_received', 'days_to_send']:
            source = 'Date received / Date sent to company'
            transform = 'temporal derivation from parsed dates'
        else:
            source = 'engineered'
            transform = 'numeric feature'
        feature_type = 'numeric'
    elif col.startswith(('Product_', 'Issue_', 'State_', 'Submitted via_')):
        source = col.split('_', 1)[0]
        transform = 'one-hot encoding from categorical source field'
        feature_type = 'categorical_encoded'
    else:
        source = 'other'
        transform = 'pass-through or engineered'
        feature_type = 'other'

    feature_dictionary_rows.append({
        'feature_name': col,
        'feature_type': feature_type,
        'source_field': source,
        'transformation': transform,
        'missing_pct': round(float(X[col].isna().mean() * 100), 4),
    })

feature_dictionary = pd.DataFrame(feature_dictionary_rows).sort_values(['feature_type', 'feature_name'])
feature_dictionary.to_csv(reports_dir / 'feature_dictionary.csv', index=False)
print('Saved feature dictionary to ../reports/feature_dictionary.csv')
display(feature_dictionary.head(20))

Saved feature dictionary to ../reports/feature_dictionary.csv


,feature_name,feature_type,source_field,transformation,missing_pct
26,Issue_APR or interest rate,categorical_encoded,Issue,one-hot encoding from categorical source field,0.0
27,"Issue_Account opening, closing, or management",categorical_encoded,Issue,one-hot encoding from categorical source field,0.0
28,Issue_Account terms and changes,categorical_encoded,Issue,one-hot encoding from categorical source field,0.0
29,Issue_Advertising,categorical_encoded,Issue,one-hot encoding from categorical source field,0.0
30,Issue_Advertising and marketing,categorical_encoded,Issue,one-hot encoding from categorical source field,0.0
31,"Issue_Advertising and marketing, including pro...",categorical_encoded,Issue,one-hot encoding from categorical source field,0.0
32,Issue_Application processing delay,categorical_encoded,Issue,one-hot encoding from categorical source field,0.0
33,"Issue_Application, originator, mortgage broker",categorical_encoded,Issue,one-hot encoding from categorical source field,0.0
34,Issue_Applying for a mortgage or refinancing a...,categorical_encoded,Issue,one-hot encoding from categorical source field,0.0
35,Issue_Arbitration,categorical_encoded,Issue,one-hot encoding from categorical source field,0.0


In [8]:
# Validate leakage boundaries and save preprocessing summaries
reports_dir = Path('../reports')
reports_dir.mkdir(parents=True, exist_ok=True)

response_col = 'Company response to consumer'
leakage_columns = [response_col, 'target_monetary_relief']
predictor_leakage_hits = [c for c in leakage_columns if c in X.columns]

print('Leakage check (predictors must not include response/target columns):')
if predictor_leakage_hits:
    print(f'  WARNING: leakage columns found in X: {predictor_leakage_hits}')
else:
    print('  OK: no leakage columns found in X')

missing_summary = pd.DataFrame({
    'feature': X.columns,
    'missing_count': X.isna().sum().values,
    'missing_pct': (X.isna().mean() * 100).round(4).values,
    'dtype': X.dtypes.astype(str).values,
}).sort_values(['missing_pct', 'missing_count'], ascending=False)

preprocess_overview = pd.DataFrame([
    {'metric': 'rows_after_target_filter', 'value': len(df)},
    {'metric': 'positive_rate_pct', 'value': round(y.mean() * 100, 4)},
    {'metric': 'feature_count', 'value': X.shape[1]},
    {'metric': 'categorical_encoded_columns', 'value': len(df_cat_encoded.columns)},
    {'metric': 'numeric_feature_count', 'value': len(numeric_features)},
    {'metric': 'narrative_length_cap_p99', 'value': round(float(narrative_len_cap), 2)},
    {'metric': 'predictor_leakage_columns_found', 'value': len(predictor_leakage_hits)},
])

preprocessing_summary = pd.concat([
    preprocess_overview,
    pd.DataFrame([{'metric': f'missing_pct::{row.feature}', 'value': row.missing_pct} for row in missing_summary.head(20).itertuples(index=False)])
], ignore_index=True)

preprocessing_summary.to_csv(reports_dir / 'preprocessing_summary.csv', index=False)
print('Saved preprocessing summary to ../reports/preprocessing_summary.csv')
display(preprocessing_summary.head(20))

Leakage check (predictors must not include response/target columns):
  OK: no leakage columns found in X
Saved preprocessing summary to ../reports/preprocessing_summary.csv


,metric,value
0,rows_after_target_filter,48499.0000
1,positive_rate_pct,1.3856
2,feature_count,250.0000
3,categorical_encoded_columns,243.0000
4,numeric_feature_count,6.0000
5,narrative_length_cap_p99,3138.2200
6,predictor_leakage_columns_found,0.0000
7,missing_pct::narrative_length,0.0000
8,missing_pct::has_narrative,0.0000
9,missing_pct::year_received,0.0000


## 6. Train/Test Split

Create the modeling split and persist train/test datasets for Notebook 3.


In [9]:
# Train-test split (80-20)
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train set: {X_train.shape[0]:,} rows ({y_train.mean():.2%} positive)')
print(f'Test set: {X_test.shape[0]:,} rows ({y_test.mean():.2%} positive)')

# Save train/test split for modeling notebooks
train_data = X_train.copy()
train_data['target'] = y_train.values
test_data = X_test.copy()
test_data['target'] = y_test.values

train_data.to_csv('../data/processed/train_features.csv', index=False)
test_data.to_csv('../data/processed/test_features.csv', index=False)

print(f'\nSaved train/test splits to data/processed/')

Train set: 38,799 rows (1.39% positive)
Test set: 9,700 rows (1.38% positive)

Saved train/test splits to data/processed/


## 7. Notes and Handoff

This preprocessing flow is designed for reproducibility and local memory limits. It uses a sampled working dataset, so absolute rates can shift versus the full raw file, but transformation logic and leakage controls remain the same.

Response-derived fields are used only for target construction and are excluded from predictors. Text remains as cleaned raw strings in exported features and is vectorized in Notebook 3.


## 8. Artifact Check

Verify the expected preprocessing outputs exist before moving into modeling.


In [10]:
# Quick artifact checklist
expected_outputs = [
    Path('../data/processed/train_features.csv'),
    Path('../data/processed/test_features.csv'),
    Path('../reports/preprocessing_summary.csv'),
    Path('../reports/feature_dictionary.csv'),
]

artifact_status = pd.DataFrame({
    'artifact': [str(p) for p in expected_outputs],
    'exists': [p.exists() for p in expected_outputs],
})
display(artifact_status)
print(f"Artifacts present: {artifact_status['exists'].sum()} / {len(artifact_status)}")

,artifact,exists
0,../data/processed/train_features.csv,True
1,../data/processed/test_features.csv,True
2,../reports/preprocessing_summary.csv,True
3,../reports/feature_dictionary.csv,True


Artifacts present: 4 / 4


## 9. Completion Summary

Summarize the final dataset handoff to the modeling notebook.


In [11]:
# Summary
print("\n" + "="*60)
print("PREPROCESSING COMPLETE")
print("="*60)
print(f"Target: Monetary relief (binary classification)")
print(f"Training samples: {len(X_train):,}")
print(f"Test samples: {len(X_test):,}")
print(f"Features: {X.shape[1]}")
print(f"  - Narratives with text: {X['narrative_clean'].str.len().gt(0).sum()}")
print(f"  - Categorical features encoded: {len(df_cat_encoded.columns)}")
print(f"  - Numeric/temporal features: {len(numeric_features)}")
print(f"\nNext: Run 03_modeling.ipynb for model comparison")


PREPROCESSING COMPLETE
Target: Monetary relief (binary classification)
Training samples: 38,799
Test samples: 9,700
Features: 250
  - Narratives with text: 12723
  - Categorical features encoded: 243
  - Numeric/temporal features: 6

Next: Run 03_modeling.ipynb for model comparison
